# Etapa 04 — Gold e features temporais

Atualização de 2026-10-07. Estado: construção e verificação reais informadas pelo autor;
**revisão `14ab57e` publicada com CI aprovada e 104 testes**. O baseline de código é a EDA
[`fc22a48`](https://github.com/wanderson42/fraud-detection-mlops/commit/fc22a48af5e6d9a0f9648a62effa95d39e02de02),
com [CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37634000505).

Referências: [contrato Gold](../../docs/GOLD_CONTRACT.md), [protocolo](../../docs/EVALUATION_PROTOCOL.md)
e [recibo de preparação](../../references/evidence/gold_preparation_2026-10-07.json).
Esta etapa prepara dados para modelagem, sem treinar nem avaliar o teste final.


## Hipóteses da EDA e decisões

A EDA do treino informou 268.668 transações, 1.505 fraudes (0,5602%) e quatro valores zero,
com evidência no [recibo](../../references/evidence/eda_training_2026-10-07.json).
A diferença de valores entre classes, o histórico curto de alguns clientes e a dinâmica
por terminal motivam candidatos de volume/valor/risco histórico. Sua eficácia ainda será comparada.

A Gold contém 19 preditores: valor e calendário atuais (3), histórico/razões de cliente (8),
volume e rótulos conhecidos do terminal (8). IDs são metadados, o alvo fica separado e
`TX_FRAUD_SCENARIO` é excluído. As janelas iniciais são de 1 e 7 dias; não há ajuste aprendido.


## Passado estrito e feedback

Para uma transação em `t`, o comportamento usa `[t − janela, t)`. Todos os peers com timestamp
igual a `t` são excluídos: IDs não comprovam a ordem de chegada.

Rótulos ficam disponíveis em `timestamp + 7 dias`. O risco usa transações de
`[t − 7 dias − janela, t − 7 dias)`, com numerador e denominador elegíveis no mesmo intervalo.
A disponibilidade precisa ser estritamente anterior a `t`, inclusive nos testes com nanossegundos.

O contexto inclui os gaps entre as janelas, mas somente treino/validação/teste são publicados.
No protocolo atual, o contexto é 1 de abril a 26 de maio; nenhum dado posterior entra na Gold.
O começo do snapshot tem histórico parcial e todos os eventos são preservados.


## Primeiro evento e média zero

Sem histórico, contagens/médias/taxas usam zero; contadores explicitam o suporte.
A razão de valor só é válida quando a média anterior é positiva; em outro caso, razão zero e
flag de validade zero. Uma média histórica zero com contagem positiva pode corresponder
a pagamentos anteriores de valor zero. Essas regras fixas não aprendem valores dos holdouts.

O modelo futuro receberá exclusivamente `X`, extraído pela allowlist. Metadados incluem IDs,
timestamps, disponibilidade de rótulo e alvo; não devem ser concatenados ao modelo.


## Construção e verificação local

Na raiz do checkout:

```bash
make validate
poetry run python -m fraud_detection_mlops.gold build
poetry run python -m fraud_detection_mlops.gold verify
```

O resultado esperado em cobertura é 42 partições e 19 preditores. As contagens reais por split
serão reconciliadas durante a construção. Preserve `gold_path`, `audit_path`, manifesto e Parquets.
Os aliases são `make gold` e `make verify-gold`. As células abaixo leem a Gold construída;
outputs permanecem vazios até uma execução real do autor.


In [1]:
from fraud_detection_mlops.gold import verify_gold

print(verify_gold())

2026-10-07 13:15:21.137 | INFO     | fraud_detection_mlops.config:<module>:11 - PROJ_ROOT path is: /home/wanderson/Documentos/fraud-detection-mlops


{'gold_path': '/home/wanderson/Documentos/fraud-detection-mlops/data/processed/handbook/6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a/gold_v1', 'verified_partitions': 42, 'feature_count': 19, 'split_rows': {'train': 268668, 'validation': 67255, 'test': 66954}, 'rows': 402877}


In [3]:
from fraud_detection_mlops.gold import load_gold_split

X, y, metadata = load_gold_split("train")
print("Shapes:", X.shape, y.shape, metadata.shape)
print("Preditores:", list(X.columns))
print("Target:", y.name)
print("Tipos:", X.dtypes.to_string())

Shapes: (268668, 19) (268668,) (268668, 6)
Preditores: ['TX_AMOUNT', 'TX_HOUR', 'TX_WEEKDAY', 'CUSTOMER_TX_COUNT_1D', 'CUSTOMER_AVG_AMOUNT_1D', 'CUSTOMER_AMOUNT_RATIO_1D', 'CUSTOMER_AMOUNT_RATIO_VALID_1D', 'CUSTOMER_TX_COUNT_7D', 'CUSTOMER_AVG_AMOUNT_7D', 'CUSTOMER_AMOUNT_RATIO_7D', 'CUSTOMER_AMOUNT_RATIO_VALID_7D', 'TERMINAL_TX_COUNT_1D', 'TERMINAL_KNOWN_LABEL_COUNT_1D', 'TERMINAL_KNOWN_FRAUD_COUNT_1D', 'TERMINAL_KNOWN_FRAUD_RATE_1D', 'TERMINAL_TX_COUNT_7D', 'TERMINAL_KNOWN_LABEL_COUNT_7D', 'TERMINAL_KNOWN_FRAUD_COUNT_7D', 'TERMINAL_KNOWN_FRAUD_RATE_7D']
Target: TX_FRAUD
Tipos: TX_AMOUNT                         float64
TX_HOUR                              int8
TX_WEEKDAY                           int8
CUSTOMER_TX_COUNT_1D                int64
CUSTOMER_AVG_AMOUNT_1D            float64
CUSTOMER_AMOUNT_RATIO_1D          float64
CUSTOMER_AMOUNT_RATIO_VALID_1D       int8
CUSTOMER_TX_COUNT_7D                int64
CUSTOMER_AVG_AMOUNT_7D            float64
CUSTOMER_AMOUNT_RATIO_7D          fl

## Validação, evidência e limites

Os testes comparam as janelas SQL a uma especificação independente por datetime,
incluindo futuro, peers, limites exatos, rótulos indisponíveis, gaps e divisão por zero.
A publicação também confere schema, hashes, IDs, datas, contagens e alvos preservados.

**Resultados reais da Gold:** construção e verificação informadas pelo autor, conforme o registro abaixo. O [recibo](../../references/evidence/gold_preparation_2026-10-07.json)
registra exclusivamente a preparação e seu baseline; o commit futuro do autor não é antecipado.
Os outputs publicados da EDA anterior permanecem no notebook daquela etapa.

Causalidade aqui é por tempo de evento, assumindo transações anteriores disponíveis.
A fonte não informa chegada tardia ou revisão: replay/streaming e paridade offline/online
exigirão contrato próprio. O loader carrega um split em memória; não é treinamento em streaming.
A primeira Gold não é feature store nem demonstra uma métrica de detecção.

As contagens e o run de auditoria foram informados. A revisão `14ab57e` e sua CI aprovada foram associadas; a saída dos checks locais do autor ainda não foi informada.
A próxima etapa é o baseline: candidatos ajustados no treino, seleção na validação e avaliação
final somente após congelar as escolhas. Transformações aprendidas ficam dentro desse pipeline.


## Gold real — execução informada em 2026-10-07

O autor forneceu build com status `success` e verify concordante: **42 partições,
19 preditores e 402.877 linhas**. Treino: 268.668; validação: 67.255; teste: 66.954.
O treino reconcilia com a EDA anterior. Run de auditoria: `8b1564df49fc497080963986b05aa007`.

O [recibo da execução](../../references/evidence/gold_build_2026-10-07.json)
transcreve o terminal e distingue essa evidência dos testes controlados. Os Parquets,
o manifesto e a auditoria nativos não foram inspecionados pelo assistente. Revisão `14ab57e` e CI aprovada foram conferidas.
A saída dos checks locais do autor ainda não foi informada. Os outputs salvos das células de verificação e leitura do treino
foram publicados pelo autor e preservados nesta atualização; o assistente não reexecutou o notebook.

Os 19 preditores são candidatos escolhidos no desenho do contrato. A seleção por
desempenho ficará no pipeline: ajuste no treino, comparação na validação e teste
final após congelar as escolhas. Não há modelo avaliado neste marco.


## Publicação e CI conferidas em 2026-10-07

Revisão [`14ab57e`](https://github.com/wanderson42/fraud-detection-mlops/commit/14ab57e15f0931ffb6966b26d390a42b514762b7),
com [CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37652509812):
104 testes em 11,37 s; lint, formatação e lockfile aprovados.
O [recibo](../../references/evidence/gold_build_2026-10-07.json) associa
a construção real informada pelo autor à revisão publicada e à evidência da CI.

O notebook da Gold publicado contém verificação concordante e leitura do treino:
`X` (268.668, 19), `y` (268.668,) e metadados (268.668, 6). Seus outputs foram
conferidos e preservados; não foram gerados pela preparação desta documentação.
A CI usa dados controlados, e os artefatos nativos da Gold continuam locais.
Nenhum modelo foi avaliado. O próximo marco é o baseline temporal.
